# Lab 9 — Transformer NMT (English → Tamil) with FastText vs. BPEmb Embeddings

Same ilci/BPCC data and the same train/val/test splits as Lab 8 (copied and checksum-verified, §1).
Three Transformer systems are trained with OpenNMT-py and scored with sacreBLEU, bin by bin:

| Arm | Tokens | Embedding initialisation |
|---|---|---|
| **A. FastText** | whitespace words, vocab capped at 32k per side | FastText wiki vectors (`wiki.en.vec`, `wiki.ta.vec`), fine-tuned |
| **B. BPEmb** | BPEmb subwords (25k SentencePiece, `vs=25000`) | BPEmb pretrained subword vectors (`dim=300`), fine-tuned |
| **C. BPEmb, random init** (control) | same subwords as B | random init, no pretrained vectors |

Arm C isolates the effect of the pretrained vectors from the effect of the tokenisation, so the
comparison between A and B can be read correctly.

All arms share the architecture: 4 encoder + 4 decoder layers, d_model = 300 (equal to the
pretrained dimension, so no vector is truncated), 4 heads, FFN 1200, and the same optimiser and
step budget. Lab 8 used d = 256, so its row in the results table is a reference, not a controlled baseline.

**Deviations from the reading of the assignment, and why** (each is measured below, not assumed):
- BPEmb's default `preprocess` lowercases *and* maps every digit to `0`. Its SentencePiece models
  were trained on that normalised text, so the raw cased or digit-bearing text fragments. We keep
  lowercasing on the English side and keep digits on both sides (§2 shows the numbers).
- Arms A and B use different tokenisations, so they differ in two things at once. Arm C is the control for this.
- English FastText uses only the first ~800 MB of `wiki.en.vec`. The file is sorted by frequency, so this
  covers the most frequent words; the rest fall back to random initialisation (§3 reports the coverage).


## 0. Setup

In [ ]:
import os, subprocess, sys
IN_COLAB = os.path.exists("/content/sample_data")
BASE = os.environ.get("LAB9_BASE", "/content/lab9" if IN_COLAB else ".")   # Colab: zip unzips to /content/lab9
DATA = f"{BASE}/data"
ONMT_VENV = "/content/onmt_venv" if IN_COLAB else "../Lab8/.onmt_venv"
ONMT = os.environ.get("LAB9_ONMT", f"{ONMT_VENV}/bin")
if IN_COLAB and not os.path.exists(f"{ONMT_VENV}/bin/onmt_train"):
    # OpenNMT-py 3.5.1 pins torch 2.1-2.2, so it gets its own venv instead of downgrading Colab's torch
    subprocess.run([sys.executable, "-m", "venv", ONMT_VENV], check=True)
    subprocess.run([f"{ONMT_VENV}/bin/pip", "install", "-q", "torch==2.2.2", "OpenNMT-py==3.5.1",
                    "bpemb", "sentencepiece", "sacrebleu", "pandas", "pyyaml", "requests"], check=True)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bpemb", "sentencepiece", "sacrebleu"], check=True)

import hashlib, json, re, shutil, time, unicodedata
from collections import Counter
from glob import glob

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import torch
import sacrebleu
from sacrebleu.metrics import BLEU
import sentencepiece as spm
from bpemb import BPEmb

SMOKE = os.environ.get("LAB9_SMOKE", "0") == "1"   # tiny end-to-end check when set
CACHE = f"{DATA}/cache"            # encodings, vocabularies, pretrained tensors (shared by smoke and full)
OUT = f"{DATA}/smoke_runs" if SMOKE else f"{DATA}/runs"
if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/lab9_runs"        # checkpoints survive a Colab disconnect
TRAIN_STEPS = 40 if SMOKE else 6000
SAVE_EVERY = 20 if SMOKE else 1000
VALID_EVERY = 20 if SMOKE else 1000
WARMUP = 10 if SMOKE else 4000
REPORT_EVERY = 10 if SMOKE else 200
WORD_VOCAB = 32000
BPE_VOCAB = 25000
DIM = 300
SEED = 42
DEFAULT_SPECIALS = ["<unk>", "<blank>", "<s>", "</s>"]
RAW_FT = os.path.expanduser("~/.cache/lab9_fasttext")   # raw FastText downloads (not in repo)
for d in (CACHE, OUT, RAW_FT):
    os.makedirs(d, exist_ok=True)
print("SMOKE =", SMOKE, "| OUT =", OUT, "| steps =", TRAIN_STEPS)


## 1. Data: the Lab 8 splits, verified

The test set is the same 2,000 sentences (500 per length bin) that Lab 8 scored, so the BLEU numbers
are directly comparable. Checksums confirm the files were copied, not regenerated.

In [ ]:
def md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

expected_md5 = json.load(open(f"{DATA}/checksums.json"))
for name, digest in expected_md5.items():
    assert md5(f"{DATA}/{name}") == digest, name
test_bins = pd.read_csv(f"{DATA}/test_bins.csv")["bin"].tolist()
n_test = sum(1 for _ in open(f"{DATA}/test.en", encoding="utf-8"))
print("all split files match the Lab 8 checksums | test sentences:", n_test, "| per bin:",
      pd.Series(test_bins).value_counts().sort_index().to_dict())


## 2. BPEmb subword encoding (follows the BPEmb repository)

BPEmb is loaded with `preprocess=False`, which turns off its whole normalisation step (lowercasing
*and* digit mapping, gated by one flag). Before choosing a configuration, we measure each option on
2,000 training sentences: the share of `<unk>` pieces, the share of single-character pieces (a sign of
fragmentation), and whether the Tamil text survives an encode/decode round trip.

In [ ]:
en_bpe = BPEmb(lang="en", vs=BPE_VOCAB, dim=DIM, preprocess=False)
ta_bpe = BPEmb(lang="ta", vs=BPE_VOCAB, dim=DIM, preprocess=False)

def piece_stats(pieces_per_sent, sp):
    n_tok = n_unk = n_char = 0
    for pcs in pieces_per_sent:
        n_tok += len(pcs)
        for p in pcs:
            n_unk += sp.PieceToId(p) == 0
            n_char += len(p.replace("▁", "")) == 1
    return {"avg_pieces_per_sent": round(n_tok / len(pieces_per_sent), 1),
            "unk_pct": round(100 * n_unk / n_tok, 2),
            "single_char_pct": round(100 * n_char / n_tok, 1)}

src_lines = [l.rstrip("\n") for l in open(f"{DATA}/train.en", encoding="utf-8")]
tgt_lines = [l.rstrip("\n") for l in open(f"{DATA}/train.ta", encoding="utf-8")]
rng = np.random.default_rng(SEED)
idx = rng.choice(len(src_lines), 2000, replace=False)
s_en = [src_lines[i] for i in idx]
s_ta = [tgt_lines[i] for i in idx]

enc_options = {
    "EN default preprocess (lower + digits->0)": en_bpe.encode([en_bpe.preprocess(s) for s in s_en]),
    "EN lower() only, digits kept  [CHOSEN]": en_bpe.encode([s.lower() for s in s_en]),
    "EN raw cased, no preprocess": en_bpe.encode(s_en),
}
ta_options = {
    "TA default preprocess (digits->0)": ta_bpe.encode([ta_bpe.preprocess(t) for t in s_ta]),
    "TA raw, digits kept  [CHOSEN]": ta_bpe.encode(s_ta),
}
enc_table = pd.DataFrame([{"option": k, **piece_stats(v, en_bpe.spm if k.startswith("EN") else ta_bpe.spm)}
                          for k, v in {**enc_options, **ta_options}.items()])

def roundtrip_exact(sents):
    ok = sum(ta_bpe.spm.DecodePieces(ta_bpe.spm.EncodeAsPieces(t)) == t for t in sents)
    return 100 * ok / len(sents)

te_src = [l.rstrip("\n") for l in open(f"{DATA}/test.en", encoding="utf-8")]
te_ref = [l.rstrip("\n") for l in open(f"{DATA}/test.ta", encoding="utf-8")]
digit_summary = {
    "test sentences containing a digit (EN)": sum(bool(re.search(r"\d", s)) for s in te_src),
    "test sentences containing a digit (TA ref)": sum(bool(re.search(r"\d", s)) for s in te_ref),
    "TA round-trip exact (% of train sample)": round(roundtrip_exact(s_ta), 1),
    "TA round-trip exact after NFKC (% of train sample)":
        round(100 * sum(ta_bpe.spm.DecodePieces(ta_bpe.spm.EncodeAsPieces(t)) == unicodedata.normalize("NFKC", t)
                        for t in s_ta) / len(s_ta), 1),
}
print(enc_table.to_string(index=False))
print(json.dumps(digit_summary, indent=2))


In [ ]:
def encode_en(lines):
    return en_bpe.encode([l.lower() for l in lines])

def encode_ta(lines):
    return ta_bpe.encode(lines)

os.makedirs(f"{CACHE}/bpe", exist_ok=True)
for split in ["train", "val", "test"]:
    for side, enc in [("en", encode_en), ("ta", encode_ta)]:
        dst = f"{CACHE}/bpe/{split}.{side}"
        if os.path.exists(dst):
            continue
        lines = [l.rstrip("\n") for l in open(f"{DATA}/{split}.{side}", encoding="utf-8")]
        with open(dst, "w", encoding="utf-8") as f:
            for toks in enc(lines):
                f.write(" ".join(toks) + "\n")
print("BPEmb-encoded files:", sorted(os.listdir(f"{CACHE}/bpe")))


## 3. FastText vectors (raw downloads cached outside the repo)

`wiki.ta.vec` is downloaded in full. `wiki.en.vec` is 6.6 GB, so only its first 800 MB is fetched
(a `Range` request). That covers the most frequent English words, because fastText sorts the file by
frequency. The truncated final line is dropped and every retained line is checked for exactly
`dim + 1` fields. The header is ignored: the partial file does not reflect the full entry count.

In [ ]:
def download(url, dst, n_bytes=None):
    if os.path.exists(dst):
        return
    headers = {"Range": f"bytes=0-{n_bytes - 1}"} if n_bytes else {}
    tmp = dst + ".part"
    with requests.get(url, headers=headers, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    os.rename(tmp, dst)

ta_raw = f"{RAW_FT}/wiki.ta.vec"
en_raw = f"{RAW_FT}/wiki.en.first800MB.vec"
download("https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.ta.vec", ta_raw)
download("https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.vec", en_raw, n_bytes=800 * 1024 * 1024)
print({p: os.path.getsize(p) for p in (ta_raw, en_raw)})


## 4. Vocabularies and the row order that the pretrained tensors must follow

Each arm gets a vocabulary built by OpenNMT's own `onmt_build_vocab`. A pretrained tensor loads
without error if it has the right number of rows, but a correctly sized tensor in the wrong order
trains silently to a worse model than random initialisation. So the row order is read back from the
checkpoint OpenNMT writes after one probe step, and the tensor is built against that list. The probe
also confirms the vocabulary size and the specials layout.

In [ ]:
def make_config(arm, src_train, tgt_train, src_val, tgt_val, src_vocab, tgt_vocab, vocab_size,
                train_steps, pre=None, save_model=None):
    cfg = {
        "data": {
            "corpus_1": {"path_src": src_train, "path_tgt": tgt_train, "weight": 1,
                         "transforms": ["filtertoolong"]},
            "valid": {"path_src": src_val, "path_tgt": tgt_val, "transforms": ["filtertoolong"]},
        },
        "save_data": f"{CACHE}/samples_{arm}",
        "src_vocab": src_vocab, "tgt_vocab": tgt_vocab,
        "src_vocab_size": vocab_size, "tgt_vocab_size": vocab_size,
        "src_seq_length": 150, "tgt_seq_length": 150,
        "save_model": save_model or f"{OUT}/{arm}/model",
        "save_checkpoint_steps": SAVE_EVERY, "keep_checkpoint": 1,
        "train_steps": train_steps, "valid_steps": VALID_EVERY,
        "warmup_steps": WARMUP, "report_every": REPORT_EVERY, "seed": SEED,
        "encoder_type": "transformer", "decoder_type": "transformer",
        "enc_layers": 4, "dec_layers": 4, "heads": 4,
        "hidden_size": DIM, "word_vec_size": DIM, "transformer_ff": 4 * DIM,
        "dropout": 0.1, "attention_dropout": 0.1, "position_encoding": True,
        "batch_type": "tokens", "batch_size": 2048, "valid_batch_size": 32, "accum_count": 4,
        "optim": "adam", "adam_beta2": 0.998, "decay_method": "noam", "learning_rate": 2.0,
        "max_grad_norm": 0, "label_smoothing": 0.1, "normalization": "tokens",
        "param_init": 0.0, "param_init_glorot": True,
        "world_size": 1, "gpu_ranks": [0],
    }
    if pre:
        cfg["pre_word_vecs_enc"], cfg["pre_word_vecs_dec"] = pre
    return cfg

def run_onmt(tool, cfg_path, log_path):
    with open(log_path, "w") as log:
        subprocess.run([f"{ONMT}/{tool}", "-config", cfg_path] + (["-n_sample", "-1"] if tool == "onmt_build_vocab" else []),
                       stdout=log, stderr=subprocess.STDOUT, check=True)

SPLIT_FILES = {
    "ft": (f"{DATA}/train.en", f"{DATA}/train.ta", f"{DATA}/val.en", f"{DATA}/val.ta", WORD_VOCAB),
    "bpe": (f"{CACHE}/bpe/train.en", f"{CACHE}/bpe/train.ta", f"{CACHE}/bpe/val.en", f"{CACHE}/bpe/val.ta", BPE_VOCAB),
}

os.makedirs(f"{CACHE}/vocab", exist_ok=True)
os.makedirs(f"{CACHE}/probe", exist_ok=True)
for arm, (s_tr, t_tr, s_va, t_va, vsize) in SPLIT_FILES.items():
    src_vocab, tgt_vocab = f"{CACHE}/vocab/{arm}.src", f"{CACHE}/vocab/{arm}.tgt"
    if not os.path.exists(tgt_vocab):
        cfg = make_config(arm, s_tr, t_tr, s_va, t_va, src_vocab, tgt_vocab, vsize, TRAIN_STEPS)
        json.dump(cfg, open(f"{CACHE}/vocab/{arm}.json", "w"), indent=2)
        run_onmt("onmt_build_vocab", f"{CACHE}/vocab/{arm}.json", f"{CACHE}/vocab/{arm}.log")
    print(arm, "vocab files:", sum(1 for _ in open(src_vocab)), "src types,",
          sum(1 for _ in open(tgt_vocab)), "tgt types")


In [ ]:
def read_vocab_order(arm, s_tr, t_tr, s_va, t_va, vsize):
    order_path = f"{CACHE}/order_{arm}.json"
    if os.path.exists(order_path):
        return json.load(open(order_path))
    ck_base = f"{CACHE}/probe/{arm}"
    cfg = make_config(arm, s_tr, t_tr, s_va, t_va, f"{CACHE}/vocab/{arm}.src", f"{CACHE}/vocab/{arm}.tgt",
                      vsize, train_steps=1, save_model=ck_base)
    cfg.update({"save_checkpoint_steps": 1, "valid_steps": 10**9, "warmup_steps": 1, "report_every": 1,
                "enc_layers": 1, "dec_layers": 1, "transformer_ff": 64, "batch_size": 512, "accum_count": 1})
    json.dump(cfg, open(f"{CACHE}/probe/{arm}.json", "w"), indent=2)
    run_onmt("onmt_train", f"{CACHE}/probe/{arm}.json", f"{CACHE}/probe/{arm}.log")
    ck_path = f"{ck_base}_step_1.pt"
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    order = {"src": ck["vocab"]["src"], "tgt": ck["vocab"]["tgt"]}
    os.remove(ck_path)
    assert order["src"][:4] == DEFAULT_SPECIALS, order["src"][:4]
    json.dump(order, open(order_path, "w"))
    return order

ORDER = {arm: read_vocab_order(arm, *SPLIT_FILES[arm]) for arm in SPLIT_FILES}
for arm, o in ORDER.items():
    print(arm, "| src rows:", len(o["src"]), "| tgt rows:", len(o["tgt"]), "| first src:", o["src"][:6])


## 5. Pretrained vectors aligned to the vocabulary

For each token in an arm's vocabulary, the vector is looked up in its pretrained source. Words or pieces
without a vector get a random row whose scale matches the pretrained vectors. Both coverage numbers are
reported. **Type coverage** is the share of vocabulary entries that got a pretrained vector. **Token
coverage** is the share of *training tokens* that did, which is what the model actually sees.

In [ ]:
def read_fasttext_subset(path, needed):
    found, n_lines, n_bad = {}, 0, 0
    with open(path, encoding="utf-8", errors="ignore") as f:
        f.readline()                       # header is not trusted for partial files
        for line in f:
            if not line.endswith("\n"):    # truncated final line from the partial download
                n_bad += 1
                break
            parts = line.split()           # FastText lines carry a trailing space
            if len(parts) != DIM + 1:
                n_bad += 1
                continue
            n_lines += 1
            w = parts[0]
            if w in needed and w not in found:
                found[w] = np.asarray(parts[1:], dtype=np.float32)
    return found, n_lines, n_bad

def cached_ft_vectors(name, path, needed):
    cache = f"{CACHE}/ft_{name}.npz"
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True)
        return dict(zip(z["words"].tolist(), z["vecs"]))
    found, n_lines, n_bad = read_fasttext_subset(path, needed)
    words = list(found)
    np.savez(cache, words=np.array(words, dtype=object), vecs=np.stack([found[w] for w in words]))
    print(f"{name}: valid lines {n_lines:,}, rejected {n_bad}, matched {len(found):,} of {len(needed):,} needed")
    return found

def ft_lookup(vecs, lowercase_fallback):
    def f(tok):
        if tok in vecs:
            return vecs[tok]
        if lowercase_fallback and tok.lower() in vecs:
            return vecs[tok.lower()]
        return None
    return f

def bpe_lookup(model):
    def f(piece):
        if piece in DEFAULT_SPECIALS:
            return None
        try:
            return model[piece]
        except KeyError:
            return None
    return f

def build_pretrained(order, lookup, seed=SEED):
    hits = {}
    for i, tok in enumerate(order):
        if tok in DEFAULT_SPECIALS:
            continue
        v = lookup(tok)
        if v is not None:
            hits[i] = v
    scale = float(np.std(np.stack(list(hits.values())))) if hits else 0.1
    W = np.random.default_rng(seed).normal(0.0, scale, size=(len(order), DIM)).astype(np.float32)
    for i, v in hits.items():
        W[i] = v
    return torch.from_numpy(W), hits

def coverage(order_list, counts, hits):
    vocab = set(order_list) - set(DEFAULT_SPECIALS)
    hit_tokens = {order_list[i] for i in hits}
    total = sum(counts.values())
    in_vocab = sum(c for t, c in counts.items() if t in vocab)
    pretrained = sum(c for t, c in counts.items() if t in hit_tokens)
    return {"type_coverage_pct": round(100 * len(hits) / len(vocab), 2),
            "token_in_vocab_pct": round(100 * in_vocab / total, 2),
            "token_with_pretrained_pct": round(100 * pretrained / total, 2)}

os.makedirs(f"{CACHE}/pretrained", exist_ok=True)
need_en = set(ORDER["ft"]["src"])
need_ta = set(ORDER["ft"]["tgt"])
ft_en_vecs = cached_ft_vectors("en", en_raw, need_en)
ft_ta_vecs = cached_ft_vectors("ta", ta_raw, need_ta)

tr_en = [l.rstrip("\n") for l in open(f"{DATA}/train.en", encoding="utf-8")]
tr_ta = [l.rstrip("\n") for l in open(f"{DATA}/train.ta", encoding="utf-8")]
ft_counts_src = Counter(t for l in tr_en for t in l.split())
ft_counts_tgt = Counter(t for l in tr_ta for t in l.split())
bpe_counts_src = Counter(t for l in open(f"{CACHE}/bpe/train.en", encoding="utf-8") for t in l.split())
bpe_counts_tgt = Counter(t for l in open(f"{CACHE}/bpe/train.ta", encoding="utf-8") for t in l.split())

emb_stats = {}
plans = {
    "ft": (ORDER["ft"]["src"], ORDER["ft"]["tgt"], ft_lookup(ft_en_vecs, True), ft_lookup(ft_ta_vecs, False),
           ft_counts_src, ft_counts_tgt),
    "bpe": (ORDER["bpe"]["src"], ORDER["bpe"]["tgt"], bpe_lookup(en_bpe), bpe_lookup(ta_bpe),
            bpe_counts_src, bpe_counts_tgt),
}
for arm, (o_src, o_tgt, lk_src, lk_tgt, c_src, c_tgt) in plans.items():
    enc_t, enc_hits = build_pretrained(o_src, lk_src)
    dec_t, dec_hits = build_pretrained(o_tgt, lk_tgt)
    torch.save(enc_t, f"{CACHE}/pretrained/{arm}_enc.pt")
    torch.save(dec_t, f"{CACHE}/pretrained/{arm}_dec.pt")
    emb_stats[arm] = {"src": coverage(o_src, c_src, enc_hits), "tgt": coverage(o_tgt, c_tgt, dec_hits)}
json.dump(emb_stats, open(f"{CACHE}/embedding_stats.json", "w"), indent=2)
pd.DataFrame({(a, s): v for a, d in emb_stats.items() for s, v in d.items()}).T


### Diagnostic: how well do the FastText vectors cover Lab 8's SentencePiece pieces?
If Lab 8's pieces matched FastText's whole-word vectors, a subword arm would be the obvious choice for FastText
too. The check below shows whether that route is viable.

In [ ]:
def sp_coverage(vocab_file, fasttext_vecs, lang_side):
    pieces = [l.split("\t")[0] for l in open(vocab_file, encoding="utf-8")]
    stripped = [p.replace("▁", "") for p in pieces if p.replace("▁", "")]
    hit = sum(1 for p in stripped if p in fasttext_vecs)
    return {"SP pieces (Lab 8)": len(stripped), "with a FastText vector": hit,
            "type coverage %": round(100 * hit / len(stripped), 1)}

sp_check = {
    "EN (Lab 8 SentencePiece 8k)": sp_coverage(f"{DATA}/en_sp.vocab", ft_en_vecs, "en"),
    "TA (Lab 8 SentencePiece 8k)": sp_coverage(f"{DATA}/ta_sp.vocab", ft_ta_vecs, "ta"),
    "EN word vocab (this arm)": {"type coverage %": emb_stats["ft"]["src"]["type_coverage_pct"]},
    "TA word vocab (this arm)": {"type coverage %": emb_stats["ft"]["tgt"]["type_coverage_pct"]},
}
pd.DataFrame(sp_check).T


## 6. Train the three Transformer arms

Each arm runs `onmt_train` on the same data and the same budget. Arms A and B load their pretrained
tensors through `pre_word_vecs_enc` / `pre_word_vecs_dec` and fine-tune them. Arm C starts from random
weights. The `onmt_train` calls are skipped if a finished checkpoint already exists, so re-running the
notebook costs minutes, not hours. Smoke mode uses 40 steps.

In [ ]:
ARMS = {
    "ft_pretrained":  {"tokens": "ft",  "pre": True},
    "bpe_pretrained": {"tokens": "bpe", "pre": True},
    "bpe_random":     {"tokens": "bpe", "pre": False},
}

def finished_checkpoint(arm):
    ck = sorted(glob(f"{OUT}/{arm}/model_step_*.pt"), key=lambda p: int(p.rsplit("_", 1)[1].split(".")[0]))
    if ck and int(ck[-1].rsplit("_", 1)[1].split(".")[0]) >= TRAIN_STEPS:
        return ck[-1]
    return None

for arm, spec in ARMS.items():
    s_tr, t_tr, s_va, t_va, vsize = SPLIT_FILES[spec["tokens"]]
    os.makedirs(f"{OUT}/{arm}", exist_ok=True)
    pre = (f"{CACHE}/pretrained/{spec['tokens']}_enc.pt", f"{CACHE}/pretrained/{spec['tokens']}_dec.pt") if spec["pre"] else None
    cfg = make_config(arm, s_tr, t_tr, s_va, t_va, f"{CACHE}/vocab/{spec['tokens']}.src",
                      f"{CACHE}/vocab/{spec['tokens']}.tgt", vsize, TRAIN_STEPS, pre=pre)
    cfg_path = f"{OUT}/{arm}/config.json"
    json.dump(cfg, open(cfg_path, "w"), indent=2)
    if finished_checkpoint(arm):
        print(arm, "already trained:", finished_checkpoint(arm))
        continue
    t0 = time.time()
    run_onmt("onmt_train", cfg_path, f"{OUT}/{arm}/train_log.txt")
    if "batch removed" in open(f"{OUT}/{arm}/train_log.txt", errors="ignore").read():
        raise RuntimeError(f"{arm}: CUDA OOM batches were dropped; lower batch_size")
    print(arm, "trained in", round((time.time() - t0) / 60, 1), "min ->", finished_checkpoint(arm))


## 7. Translate the test set

Both arms translate the 2,000 test sentences with beam 5. The FastText arm reads the raw word-level
input. The BPEmb arms read the BPEmb-encoded input, which is the same encoding used in training.
Their output pieces are turned back into text with SentencePiece's decoder (`DecodePieces`), so
scoring is on plain Tamil, not on `▁`-marked pieces.

In [ ]:
def translate(arm, src_file, out_file):
    if os.path.exists(out_file) and sum(1 for _ in open(out_file, encoding="utf-8")) == n_test:
        return
    ck = finished_checkpoint(arm)
    with open(f"{OUT}/{arm}/translate_log.txt", "w") as log:
        subprocess.run([f"{ONMT}/onmt_translate", "-model", ck, "-src", src_file, "-output", out_file,
                        "-batch_size", "32", "-batch_type", "sents", "-beam_size", "5", "-max_length", "150", "-gpu", "0"],
                       stdout=log, stderr=subprocess.STDOUT, check=True)

translate("ft_pretrained", f"{DATA}/test.en", f"{OUT}/ft_pretrained/test_pred.ta")
translate("bpe_pretrained", f"{CACHE}/bpe/test.en", f"{OUT}/bpe_pretrained/test_pred.ta")
translate("bpe_random", f"{CACHE}/bpe/test.en", f"{OUT}/bpe_random/test_pred.ta")

def load_hyps(arm):
    raw = [l.rstrip("\n") for l in open(f"{OUT}/{arm}/test_pred.ta", encoding="utf-8")]
    if arm.startswith("ft"):
        return raw
    return [ta_bpe.spm.DecodePieces(l.split()) for l in raw]

for arm in ARMS:
    h = load_hyps(arm)
    print(arm, "| example:", h[0][:120])


## 8. Bin-wise BLEU (sacreBLEU) and the diagnostics that explain it

Scores use `BLEU(tokenize='13a')`, the same as Lab 8, and the signature is printed with the result.
The primary numbers score against the raw references. A secondary column applies NFKC to both sides,
which separates the cost of Unicode normalisation in the SentencePiece pipeline from the cost of
translation quality. Diagnostics per bin: the share of **source tokens** outside the model's vocabulary
(OOV), the share of **reference tokens** the model can never output, and the share of **output tokens**
that are `<unk>`.

In [ ]:
metric = BLEU(tokenize="13a", force=True)
bins = ["bin1", "bin2", "bin3", "bin4"]
bin_range = {"bin1": "<=10", "bin2": "(10,20]", "bin3": "(20,30]", "bin4": ">30"}
ref_raw = te_ref
ref_nfkc = [unicodedata.normalize("NFKC", r) for r in te_ref]
bins_arr = np.array(test_bins)

vocab_sets = {
    "ft": (set(ORDER["ft"]["src"]), set(ORDER["ft"]["tgt"]), te_src, te_ref),
    "bpe": (set(ORDER["bpe"]["src"]), set(ORDER["bpe"]["tgt"]),
            [l.rstrip("\n") for l in open(f"{CACHE}/bpe/test.en", encoding="utf-8")],
            [l.rstrip("\n") for l in open(f"{CACHE}/bpe/test.ta", encoding="utf-8")]),
}
arm_kind = {"ft_pretrained": "ft", "bpe_pretrained": "bpe", "bpe_random": "bpe"}

results = {}
for arm in ARMS:
    kind = arm_kind[arm]
    src_set, tgt_set, src_lines_eval, ref_lines_eval = vocab_sets[kind]
    hyps = load_hyps(arm)
    hyp_tok_raw = [l.rstrip("\n") for l in open(f"{OUT}/{arm}/test_pred.ta", encoding="utf-8")]
    per_bin = {}
    for b in bins:
        sel = np.where(bins_arr == b)[0]
        h = [hyps[i] for i in sel]
        r = [ref_raw[i] for i in sel]
        rn = [ref_nfkc[i] for i in sel]
        hn = [unicodedata.normalize("NFKC", x) for x in h]
        s_raw = metric.corpus_score(h, [r])
        s_nfkc = metric.corpus_score(hn, [rn])
        src_toks = [t for i in sel for t in src_lines_eval[i].split()]
        ref_toks = [t for i in sel for t in ref_lines_eval[i].split()]
        out_toks = [t for i in sel for t in hyp_tok_raw[i].split()]
        per_bin[b] = {
            "n": int(len(sel)),
            "BLEU": round(s_raw.score, 2),
            "BLEU_nfkc": round(s_nfkc.score, 2),
            "len_ratio": round(s_raw.sys_len / s_raw.ref_len, 3),
            "src_oov_pct": round(100 * sum(t not in src_set for t in src_toks) / len(src_toks), 2),
            "ref_oov_pct": round(100 * sum(t not in tgt_set for t in ref_toks) / len(ref_toks), 2),
            "out_unk_pct": round(100 * sum(t == "<unk>" for t in out_toks) / max(len(out_toks), 1), 2),
        }
    overall = metric.corpus_score(hyps, [ref_raw])
    overall_n = metric.corpus_score([unicodedata.normalize("NFKC", x) for x in hyps], [ref_nfkc])
    per_bin["overall"] = {"n": len(hyps), "BLEU": round(overall.score, 2), "BLEU_nfkc": round(overall_n.score, 2),
                          "len_ratio": round(overall.sys_len / overall.ref_len, 3)}
    results[arm] = per_bin
    print(arm, "|", overall.format(signature=metric.get_signature()) if arm == "ft_pretrained" else f"overall BLEU {overall.score:.2f}")

with open(f"{OUT}/bleu_results.json", "w") as f:
    json.dump(results, f, indent=2)


In [ ]:
lab8 = pd.read_csv(f"{DATA}/bleu_by_bin_lab8.csv")
lab8_row = {r["bin"]: r["BLEU"] for _, r in lab8.iterrows() if r["bin"] != "overall"}
lab8_overall = float(lab8.loc[lab8["bin"] == "overall", "BLEU"].iloc[0])

label = {"ft_pretrained": "A. FastText (word, pretrained)",
         "bpe_pretrained": "B. BPEmb (subword, pretrained)",
         "bpe_random": "C. BPEmb (subword, random init)"}
rows = []
for b in bins:
    row = {"bin": f"{b} ({bin_range[b]})", "n": results["ft_pretrained"][b]["n"]}
    for arm in ARMS:
        row[label[arm]] = results[arm][b]["BLEU"]
    row["Lab 8 (ref)"] = lab8_row[b]
    rows.append(row)
overall_row = {"bin": "overall", "n": results["ft_pretrained"]["overall"]["n"]}
for arm in ARMS:
    overall_row[label[arm]] = results[arm]["overall"]["BLEU"]
overall_row["Lab 8 (ref)"] = lab8_overall
rows.append(overall_row)
bleu_table = pd.DataFrame(rows)
bleu_table.to_csv(f"{OUT}/bleu_by_bin_table.csv", index=False)
bleu_table


In [ ]:
diag_rows = []
for arm in ARMS:
    for b in bins:
        d = results[arm][b]
        diag_rows.append({"arm": label[arm], "bin": b, "BLEU": d["BLEU"], "BLEU (NFKC refs)": d["BLEU_nfkc"],
                          "len ratio": d["len_ratio"], "src OOV %": d["src_oov_pct"],
                          "ref OOV %": d["ref_oov_pct"], "output <unk> %": d["out_unk_pct"]})
diag = pd.DataFrame(diag_rows)
diag.to_csv(f"{OUT}/diagnostics_by_bin.csv", index=False)
diag


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
x = np.arange(len(bins))
w = 0.2
for k, arm in enumerate(ARMS):
    vals = [results[arm][b]["BLEU"] for b in bins]
    ax.bar(x + (k - 1) * w, vals, w, label=label[arm])
ax.bar(x + 2 * w, [lab8_row[b] for b in bins], w, label="Lab 8 (reference, d=256)", color="lightgray")
ax.set_xticks(x)
ax.set_xticklabels([f"{b}\n{bin_range[b]} words" for b in bins])
ax.set_ylabel("sacreBLEU (raw refs)")
ax.set_title("Lab 9: bin-wise BLEU by embedding arm (EN→TA, same test split)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/bleu_by_bin.png", dpi=150)
plt.show()


In [ ]:
def parse_val(arm):
    steps, acc = [], []
    cur = None
    for line in open(f"{OUT}/{arm}/train_log.txt", encoding="utf-8", errors="ignore"):
        m = re.search(r"Step\s+(\d+)/", line)
        if m:
            cur = int(m.group(1))
        m2 = re.search(r"Validation accuracy:\s*([\d.]+)", line)
        if m2 and cur is not None:
            steps.append(cur)
            acc.append(float(m2.group(1)))
    return steps, acc

fig, ax = plt.subplots(figsize=(7, 4))
for arm in ARMS:
    s, a = parse_val(arm)
    ax.plot(s, a, marker="o", label=label[arm])
ax.set_xlabel("training step")
ax.set_ylabel("validation token accuracy (%)")
ax.set_title("Validation accuracy during training")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/validation_curves.png", dpi=150)
plt.show()


In [ ]:
for b in ["bin1", "bin4"]:
    print(f"===== {b} =====")
    sel = [i for i, bb in enumerate(test_bins) if bb == b][:2]
    for i in sel:
        print("SRC :", te_src[i])
        print("REF :", te_ref[i])
        for arm in ARMS:
            print(f"{arm:15s}:", load_hyps(arm)[i])
        print()


## 9. Results and analysis

*(Filled in after the full run — see the results table in §8 and the diagnostics in §8.)*
